# 09 – Adaptation & Fairness (RQ3)

This notebook tests:
1. Concept drift / distribution shift robustness
2. Fairness metrics (especially for thin-file applicants)
3. Regulatory-style constraints

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X = np.load(DATA_PROCESSED / "X_fused.npy")
y = np.load(DATA_PROCESSED / "y.npy")
thin = np.load(DATA_PROCESSED / "thin.npy")

np.random.seed(42)
shift_mask = np.random.rand(len(y)) < 0.35
y_shifted = y.copy()
y_shifted[shift_mask] = 1
X_shifted = X + np.random.normal(0, 0.15, X.shape).astype(np.float32)

In [ ]:
class MultiAgentCoordinator(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.risk_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.fairness_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.portfolio_agent = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.combine = nn.Sequential(nn.Linear(6, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x):
        r = self.risk_agent(x)
        f = self.fairness_agent(x)
        p = self.portfolio_agent(x)
        return self.combine(torch.cat([r, f, p], dim=-1))

marl = MultiAgentCoordinator(X.shape[1]).to(device)
marl_path = RESULTS / "multi_agent_coordinator.pt"
if marl_path.exists():
    marl.load_state_dict(torch.load(marl_path, map_location=device))
marl.eval()

In [ ]:
def evaluate_under_shift(model, X_data, y_data, thin_data):
    preds = []
    with torch.no_grad():
        for i in range(len(X_data)):
            state = torch.tensor(X_data[i], dtype=torch.float32, device=device)
            logits = model(state)
            action = logits.argmax().item()
            preds.append(action)
    preds = np.array(preds)
    
    overall_auc = roc_auc_score(y_data, preds) if len(np.unique(preds)) > 1 else 0.5
    thin_mask = thin_data == 1
    thick_mask = thin_data == 0
    
    thin_approval = (preds[thin_mask] == 0).mean() if thin_mask.sum() > 0 else 0
    thick_approval = (preds[thick_mask] == 0).mean() if thick_mask.sum() > 0 else 0
    parity_gap = abs(thin_approval - thick_approval)
    
    return {
        "AUC": overall_auc,
        "Thin-file Approval": thin_approval,
        "Thick-file Approval": thick_approval,
        "Parity Gap": parity_gap
    }

print("=== Original Distribution ===")
orig = evaluate_under_shift(marl, X, y, thin)
print(orig)

print("\n=== After Distribution Shift ===")
shifted = evaluate_under_shift(marl, X_shifted, y_shifted, thin)
print(shifted)

In [ ]:
fairness_df = pd.DataFrame([
    {"Setting": "Original", **orig},
    {"Setting": "Shifted", **shifted}
])
print("\nFairness & Adaptation Results:")
print(fairness_df.round(3).to_string(index=False))
fairness_df.to_csv(RESULTS / "adaptation_fairness.csv", index=False)
print("\nSaved → results/adaptation_fairness.csv")